<a href="https://colab.research.google.com/github/wissbendidi/domain-llm/blob/main/src/evaluation/v1_1_llm_judge_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
# Install any missing dependencies
!pip install transformers datasets scipy matplotlib seaborn openpyxl

# Clone your project or upload files
from google.colab import files
import os

# Create project structure
!mkdir -p src/evaluation
!mkdir -p evaluation_results/llm_judge
!mkdir -p data

print("✅ Colab environment ready!")

✅ Colab environment ready!


In [11]:
from google.colab import files

print("📂 Upload your v1_1_evaluation_results.csv file:")
uploaded = files.upload()

# Move uploaded file to correct location
import shutil
import os

for filename in uploaded.keys():
    if filename.endswith('.csv'):
        shutil.move(filename, 'evaluation_results/v1_1_evaluation_results.csv')
        print(f"✅ Uploaded: {filename}")
        break

# Verify upload
if os.path.exists('evaluation_results/v1_1_evaluation_results.csv'):
    print("✅ File uploaded successfully!")

    # Quick preview
    import pandas as pd
    df = pd.read_csv('evaluation_results/v1_1_evaluation_results.csv')
    print(f"📊 Loaded {len(df)} test cases")
    print("\n👀 First 3 rows:")
    print(df.head(3))
else:
    print("❌ File upload failed")

📂 Upload your v1_1_evaluation_results.csv file:


Saving v11_evaluation_results.csv to v11_evaluation_results (1).csv
✅ Uploaded: v11_evaluation_results (1).csv
✅ File uploaded successfully!
📊 Loaded 150 test cases

👀 First 3 rows:
                                  business  expected         generated  \
0    a movement to inspire positive change  ahub.com  MoveMobility.com   
1                     a hub for innovators  ahub.com    InnovateHub.co   
2  a next-generation solution for everyone  ahub.com     GreenWise.app   

   is_valid  has_artifacts  is_safety_block  similarity  
0      True          False            False    0.083333  
1      True          False            False    0.363636  
2      True          False            False    0.000000  


In [12]:
# Create the LLM Judge implementation
llm_judge_code = '''
import torch
import json
import time
import logging
from typing import Dict, List, Optional
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import re
from dataclasses import dataclass

@dataclass
class JudgeResult:
    relevance: float
    memorability: float
    brandability: float
    technical_quality: float
    creativity: float
    commercial_viability: float
    overall_score: float
    reasoning: str
    improvement_suggestions: str
    confidence: float = 0.0

class FreeLLMJudge:
    def __init__(self, model_name: str = "microsoft/DialoGPT-medium"):
        self.model_name = model_name
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"🤖 Initializing LLM Judge with {model_name} on {self.device}")
        self._initialize_model()

    def _initialize_model(self):
        try:
            self.generator = pipeline(
                "text-generation",
                model=self.model_name,
                device=0 if self.device == "cuda" else -1,
                torch_dtype=torch.float16 if self.device == "cuda" else torch.float32
            )
            self.tokenizer = self.generator.tokenizer
            print("✅ Model loaded successfully!")
        except Exception as e:
            print(f"❌ Model loading failed: {e}")
            raise

    def evaluate_domain(self, business_description: str, generated_domain: str) -> JudgeResult:
        prompt = f"""You are an expert domain name evaluator. Evaluate this domain name for the given business.

Business: "{business_description}"
Domain: "{generated_domain}"

Rate each criterion from 1-10:
1. RELEVANCE: How well does the domain relate to the business?
2. MEMORABILITY: Is it easy to remember?
3. BRANDABILITY: Would this make a good brand?
4. TECHNICAL_QUALITY: Proper format and length?
5. CREATIVITY: Is it creative and distinctive?
6. COMMERCIAL_VIABILITY: Would this work for business?

Response format:
{{"relevance": 7, "memorability": 8, "brandability": 6, "technical_quality": 9, "creativity": 5, "commercial_viability": 7, "overall_score": 7, "reasoning": "Brief explanation"}}

Evaluation:"""

        try:
            response = self.generator(
                prompt,
                max_length=len(prompt.split()) + 150,
                temperature=0.1,
                do_sample=True,
                pad_token_id=self.tokenizer.eos_token_id
            )

            generated_text = response[0]['generated_text'][len(prompt):].strip()
            result = self._parse_response(generated_text)

            if result:
                return result
            else:
                return self._create_fallback_result(business_description, generated_domain)

        except Exception as e:
            print(f"⚠️ Evaluation failed for {generated_domain}: {e}")
            return self._create_fallback_result(business_description, generated_domain)

    def _parse_response(self, response: str) -> Optional[JudgeResult]:
        try:
            # Find JSON in response
            json_match = re.search(r'\\{.*?\\}', response, re.DOTALL)
            if json_match:
                data = json.loads(json_match.group())

                # Ensure all required fields
                required_fields = ['relevance', 'memorability', 'brandability', 'technical_quality', 'creativity', 'commercial_viability']
                for field in required_fields:
                    if field not in data:
                        data[field] = 5.0
                    data[field] = max(1, min(10, float(data[field])))

                if 'overall_score' not in data:
                    scores = [data[field] for field in required_fields]
                    data['overall_score'] = sum(scores) / len(scores)

                return JudgeResult(
                    relevance=data['relevance'],
                    memorability=data['memorability'],
                    brandability=data['brandability'],
                    technical_quality=data['technical_quality'],
                    creativity=data['creativity'],
                    commercial_viability=data['commercial_viability'],
                    overall_score=data['overall_score'],
                    reasoning=data.get('reasoning', 'No reasoning provided'),
                    improvement_suggestions=data.get('improvement_suggestions', 'No suggestions provided'),
                    confidence=0.8
                )
        except:
            pass
        return None

    def _create_fallback_result(self, business_description: str, generated_domain: str) -> JudgeResult:
        # Simple rule-based fallback
        domain_clean = generated_domain.lower().replace('.com', '').replace('.io', '').replace('.ai', '')
        business_words = business_description.lower().split()

        relevance = 7.0 if any(word in domain_clean for word in business_words[:3]) else 4.0
        technical_quality = 8.0 if len(domain_clean) <= 15 and '.' in generated_domain else 5.0
        memorability = max(3.0, 10.0 - len(domain_clean) * 0.3)

        return JudgeResult(
            relevance=relevance,
            memorability=memorability,
            brandability=6.0,
            technical_quality=technical_quality,
            creativity=5.0,
            commercial_viability=6.0,
            overall_score=(relevance + memorability + technical_quality + 17.0) / 6.0,
            reasoning="Fallback evaluation",
            improvement_suggestions="Manual review recommended",
            confidence=0.3
        )

    def batch_evaluate(self, test_cases: List[Dict]) -> List[Dict]:
        results = []
        total = len(test_cases)

        for i, case in enumerate(test_cases):
            if i % 10 == 0:
                print(f"Progress: {i}/{total} ({i/total*100:.1f}%)")

            business = case.get('business', '')
            generated = case.get('generated', '')

            evaluation = self.evaluate_domain(business, generated)

            case_with_eval = case.copy()
            case_with_eval['llm_evaluation'] = {
                'relevance': evaluation.relevance,
                'memorability': evaluation.memorability,
                'brandability': evaluation.brandability,
                'technical_quality': evaluation.technical_quality,
                'creativity': evaluation.creativity,
                'commercial_viability': evaluation.commercial_viability,
                'overall_score': evaluation.overall_score,
                'reasoning': evaluation.reasoning,
                'improvement_suggestions': evaluation.improvement_suggestions,
                'confidence': evaluation.confidence
            }

            results.append(case_with_eval)
            time.sleep(0.1)  # Small delay

        print(f"✅ Completed {len(results)} evaluations!")
        return results
'''

# Save the code to file
with open('src/evaluation/llm_judge.py', 'w') as f:
    f.write(llm_judge_code)

print("✅ LLM Judge code created!")

✅ LLM Judge code created!


In [13]:
import sys
sys.path.append('/content/src')
import pandas as pd

# Import our LLM Judge
exec(open('/content/src/evaluation/llm_judge.py').read())

print("🚀 Starting LLM-as-a-Judge Evaluation for v1.1 Model")
print("="*60)

# Load your v1.1 results
df = pd.read_csv('/content/evaluation_results/v1_1_evaluation_results.csv')
print(f"📊 Loaded {len(df)} test cases from your v1.1 model")

# Convert to list of dictionaries
test_cases = []
for _, row in df.iterrows():
    test_cases.append({
        'business': row['business'],
        'expected': row['expected'],
        'generated': row['generated'],
        'is_valid': row['is_valid'],
        'similarity': row['similarity']
    })

# Initialize LLM Judge
judge = FreeLLMJudge(model_name="microsoft/DialoGPT-medium")

# Run evaluation
print(f"\n🔍 Evaluating {len(test_cases)} domains from v1.1 model...")
enhanced_results = judge.batch_evaluate(test_cases)

print("✅ LLM Judge evaluation for v1.1 completed!")

🚀 Starting LLM-as-a-Judge Evaluation for v1.1 Model
📊 Loaded 150 test cases from your v1.1 model
🤖 Initializing LLM Judge with microsoft/DialoGPT-medium on cpu


Device set to use cpu
Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Both `max_new_tokens` (=256) and `max_length`(=245) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ Model loaded successfully!

🔍 Evaluating 150 domains from v1.1 model...
Progress: 0/150 (0.0%)


Both `max_new_tokens` (=256) and `max_length`(=243) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=244) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=245) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=245) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Progress: 10/150 (6.7%)


...

Progress: 130/150 (86.7%)


Both `max_new_tokens` (=256) and `max_length`(=254) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=253) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=258) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=257) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Progress: 140/150 (93.3%)


Both `max_new_tokens` (=256) and `max_length`(=246) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=246) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=246) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=245) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ Completed 150 evaluations!
✅ LLM Judge evaluation for v1.1 completed!


In [14]:
# Display summary statistics for v1.1
print("📈 v1.1 MODEL - LLM JUDGE EVALUATION RESULTS")
print("="*60)

# Basic stats
total_cases = len(enhanced_results)
valid_domains = sum(1 for r in enhanced_results if r.get('is_valid', False))
validity_rate = valid_domains / total_cases * 100

similarity_scores = [r.get('similarity', 0) for r in enhanced_results]
avg_similarity = sum(similarity_scores) / len(similarity_scores)

print(f"📊 Total test cases: {total_cases}")
print(f"✅ Valid domains: {valid_domains} ({validity_rate:.1f}%)")
print(f"🎯 Average similarity: {avg_similarity:.3f}")

# LLM Judge stats
llm_results = [r['llm_evaluation'] for r in enhanced_results]
metrics = ['relevance', 'memorability', 'brandability', 'technical_quality', 'creativity', 'commercial_viability', 'overall_score']

print(f"\n🤖 LLM JUDGE SCORES FOR v1.1:")
print("-" * 40)
for metric in metrics:
    scores = [lr[metric] for lr in llm_results]
    avg_score = sum(scores) / len(scores)
    print(f"📊 {metric.replace('_', ' ').title():<20}: {avg_score:.1f}/10")

# Show best and worst examples
print(f"\n🏆 BEST v1.1 EXAMPLES:")
sorted_results = sorted(enhanced_results, key=lambda x: x['llm_evaluation']['overall_score'], reverse=True)
for i, result in enumerate(sorted_results[:3]):
    llm_eval = result['llm_evaluation']
    print(f"{i+1}. {result['generated']} -> {llm_eval['overall_score']:.1f}/10")
    print(f"   Business: {result['business']}")
    print(f"   Reasoning: {llm_eval['reasoning'][:100]}...")
    print()

print(f"\n👎 WORST v1.1 EXAMPLES:")
for i, result in enumerate(sorted_results[-3:]):
    llm_eval = result['llm_evaluation']
    print(f"{i+1}. {result['generated']} -> {llm_eval['overall_score']:.1f}/10")
    print(f"   Business: {result['business']}")
    print(f"   Reasoning: {llm_eval['reasoning'][:100]}...")
    print()

📈 v1.1 MODEL - LLM JUDGE EVALUATION RESULTS
📊 Total test cases: 150
✅ Valid domains: 122 (81.3%)
🎯 Average similarity: 0.164

🤖 LLM JUDGE SCORES FOR v1.1:
----------------------------------------
📊 Relevance           : 5.5/10
📊 Memorability        : 6.3/10
📊 Brandability        : 6.0/10
📊 Technical Quality   : 7.4/10
📊 Creativity          : 5.0/10
📊 Commercial Viability: 6.0/10
📊 Overall Score       : 6.0/10

🏆 BEST v1.1 EXAMPLES:
1. FinAPIs.io -> 6.6/10
   Business: a multi-cloud API gateway for fintech applications
   Reasoning: Fallback evaluation...

2. GDPRlaw.com -> 6.6/10
   Business: a consultancy specializing in GDPR and CCPA compliance
   Reasoning: Fallback evaluation...

3. kubeseal.com -> 6.6/10
   Business: a container security platform for Kubernetes clusters
   Reasoning: Fallback evaluation...


👎 WORST v1.1 EXAMPLES:
1. SeaNest.coLabs<|endoftext| -> 4.8/10
   Business: Our startup produces lab-grown seafood to combat overfishing and reduce the environmental impact of

In [15]:
# Save enhanced v1.1 results
import json

# Save as JSON
with open('/content/evaluation_results/v1_1_with_llm_judge.json', 'w') as f:
    json.dump(enhanced_results, f, indent=2)

# Save as CSV for easy viewing
flattened_data = []
for result in enhanced_results:
    flat_result = {
        'business': result['business'],
        'expected': result['expected'],
        'generated': result['generated'],
        'is_valid': result['is_valid'],
        'similarity': result['similarity'],
        'llm_relevance': result['llm_evaluation']['relevance'],
        'llm_memorability': result['llm_evaluation']['memorability'],
        'llm_brandability': result['llm_evaluation']['brandability'],
        'llm_technical_quality': result['llm_evaluation']['technical_quality'],
        'llm_creativity': result['llm_evaluation']['creativity'],
        'llm_commercial_viability': result['llm_evaluation']['commercial_viability'],
        'llm_overall_score': result['llm_evaluation']['overall_score'],
        'llm_confidence': result['llm_evaluation']['confidence'],
        'llm_reasoning': result['llm_evaluation']['reasoning'][:200]  # Truncated for CSV
    }
    flattened_data.append(flat_result)

df_enhanced = pd.DataFrame(flattened_data)
df_enhanced.to_csv('/content/evaluation_results/v1_1_with_llm_judge.csv', index=False)

print("✅ v1.1 Results saved!")
print("📁 Files created:")
print("   - v1_1_with_llm_judge.json")
print("   - v1_1_with_llm_judge.csv")

✅ v1.1 Results saved!
📁 Files created:
   - v1_1_with_llm_judge.json
   - v1_1_with_llm_judge.csv


In [16]:
# Download the enhanced v1.1 results
from google.colab import files

print("📥 Downloading enhanced v1.1 results...")
files.download('/content/evaluation_results/v1_1_with_llm_judge.csv')
files.download('/content/evaluation_results/v1_1_with_llm_judge.json')

print("✅ Download complete!")
print("\n🎉 LLM-as-a-Judge evaluation for v1.1 model finished!")
print("\n📋 For your technical report, you now have:")
print("   - v1.1 model results with LLM judge scores")
print("   - 6 detailed metrics for model comparison")
print("   - Overall LLM judge scores for v1.1 performance")
print("   - Detailed reasoning for each v1.1 domain evaluation")

📥 Downloading enhanced v1.1 results...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Download complete!

🎉 LLM-as-a-Judge evaluation for v1.1 model finished!

📋 For your technical report, you now have:
   - v1.1 model results with LLM judge scores
   - 6 detailed metrics for model comparison
   - Overall LLM judge scores for v1.1 performance
   - Detailed reasoning for each v1.1 domain evaluation
